In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import json

dbutils.widgets.text("cutoff_ts", "2017-12-31 23:59:59")
dbutils.widgets.text("full_reload", "false")
CUTOFF = dbutils.widgets.get("cutoff_ts")
FULL = dbutils.widgets.get("full_reload").lower() == "true"

CATALOG = "olist_dev"
S = f"{CATALOG}.silver"
BRONZE = "abfss://bronze@stolistdl2026.dfs.core.windows.net/olist/"
CTRL = f"{S}.etl_watermark"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CTRL} (
  table_name STRING, last_watermark TIMESTAMP,
  last_run_ts TIMESTAMP, rows_loaded BIGINT)
""")

def get_wm(name):
    r = spark.table(CTRL).filter(F.col("table_name") == name).first()
    return str(r["last_watermark"]) if r and r["last_watermark"] else "1900-01-01 00:00:00"

def read_csv(fname):
    return (spark.read.option("header", True).option("multiLine", True)
            .option("quote", '"').option("escape", '"').csv(BRONZE + fname))

def upsert(df, table, keys):
    if not spark.catalog.tableExists(table):
        df.limit(0).write.format("delta").saveAsTable(table)
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
ts_cols = ["order_purchase_timestamp", "order_approved_at",
           "order_delivered_carrier_date", "order_delivered_customer_date",
           "order_estimated_delivery_date"]

orders = read_csv("olist_orders_dataset.csv")
for c in ts_cols:
    orders = orders.withColumn(c, F.to_timestamp(c))
orders = orders.dropDuplicates(["order_id"])

WM = "1900-01-01 00:00:00" if FULL else get_wm("orders")
print("Old watermark:", WM, "| Cutoff:", CUTOFF, "| Full reload:", FULL)

new_orders = (orders
    .filter((F.col("order_purchase_timestamp") > F.lit(WM).cast("timestamp")) &
            (F.col("order_purchase_timestamp") <= F.lit(CUTOFF).cast("timestamp")))
    .withColumn("ingestion_ts", F.current_timestamp()))

n_orders = new_orders.count()
print("New orders to load:", n_orders)

Old watermark: 2017-12-31 23:29:31 | Cutoff: 2017-12-31 23:59:59 | Full reload: False
New orders to load: 0


In [0]:
n_items = 0
if n_orders > 0:
    # 1. Orders upsert
    upsert(new_orders, f"{S}.orders_inc", ["order_id"])

    # 2. Unhi orders ke items
    items = (read_csv("olist_order_items_dataset.csv")
        .withColumn("order_item_id", F.col("order_item_id").cast("int"))
        .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
        .withColumn("price", F.col("price").cast("double"))
        .withColumn("freight_value", F.col("freight_value").cast("double"))
        .dropDuplicates(["order_id", "order_item_id"])
        .join(new_orders.select("order_id"), "order_id", "left_semi")
        .withColumn("ingestion_ts", F.current_timestamp()))
    n_items = items.count()
    upsert(items, f"{S}.order_items_inc", ["order_id", "order_item_id"])

    # 3. Watermark SIRF successful load ke baad update
    new_wm = str(new_orders.agg(F.max("order_purchase_timestamp")).first()[0])
    spark.sql(f"""
      MERGE INTO {CTRL} t
      USING (SELECT 'orders' AS table_name, timestamp'{new_wm}' AS last_watermark,
                    current_timestamp() AS last_run_ts, CAST({n_orders} AS BIGINT) AS rows_loaded) s
      ON t.table_name = s.table_name
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *""")
    print("New watermark:", new_wm)
else:
    print("Koi naya data nahi, skip.")

print(f"Loaded -> orders: {n_orders}, items: {n_items}")
dbutils.notebook.exit(json.dumps({"orders_loaded": n_orders, "items_loaded": n_items}))

In [0]:
%sql
SELECT 'orders_inc' t, COUNT(*) c, COUNT(DISTINCT order_id) d FROM olist_dev.silver.orders_inc
UNION ALL
SELECT 'items_inc', COUNT(*), COUNT(DISTINCT concat(order_id, order_item_id)) FROM olist_dev.silver.order_items_inc;

t,c,d
orders_inc,45430,45430
items_inc,51234,51234
